In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [26]:
!pip install tensorflow opencv-python matplotlib


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from tensorflow.keras import layers, models

In [28]:
IMAGE_SIZE = 256
NUM_CLASSES = 7   
BATCH_SIZE = 8
EPOCHS = 20


In [29]:
COLOR_MAP = {
    (0, 0, 0): 0,
    (255, 0, 0): 1,
    (255, 255, 0): 2,
    (0, 0, 255): 3,
    (128, 128, 128): 4,
    (0, 255, 0): 5,
    (128, 0, 0): 6
}

In [30]:
CLASS_NAMES = [
    "Background",
    "Building",
    "Road",
    "Water",
    "Barren land",
    "Forest / Green",
    "Agriculture"
]

PALETTE = np.array([
    [0, 0, 0],        # 0 Background
    [255, 0, 0],      # 1 Building
    [255, 255, 0],    # 2 Road
    [0, 0, 255],      # 3 Water
    [128, 128, 128],  # 4 Barren
    [0, 255, 0],      # 5 Forest
    [128, 0, 0],      # 6 Agriculture
], dtype=np.uint8)





In [31]:
def load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Image not found: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    img = img / 255.0
    return img.astype(np.float32)

In [32]:
def load_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Mask not found: {path}")

    mask = cv2.resize(
        mask,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    mask = mask.astype(np.int32)

    # Ignore no-data (0)
    # Convert 1–7 → 0–6
    valid_mask = mask > 0
    mask[valid_mask] = mask[valid_mask] - 1

    return mask


In [33]:
mask = load_mask("/kaggle/input/loveda-dataset/Val/Val/Urban/masks_png/3517.png")
print(np.unique(mask))


ValueError: Mask not found: /kaggle/input/loveda-dataset/Val/Val/Urban/masks_png/3517.png

In [34]:
def data_generator(image_dirs, mask_dirs):
    for img_dir, msk_dir in zip(image_dirs, mask_dirs):
        images = sorted(os.listdir(img_dir))
        for img_name in images:
            image = load_image(os.path.join(img_dir, img_name))
            mask = load_mask(os.path.join(msk_dir, img_name))

            # weight = 0 for no-data pixels
            sample_weight = (mask >= 0).astype(np.float32)
            sample_weight[mask == 0] = 0.0

            yield image, mask, sample_weight

In [35]:
BASE_PATH = "/kaggle/input/loveda-dataset"

train_image_dirs = [
    f"{BASE_PATH}/Train/Train/Urban/images_png",
    f"{BASE_PATH}/Train/Train/Rural/images_png"
]

train_mask_dirs = [
    f"{BASE_PATH}/Train/Train/Urban/masks_png",
    f"{BASE_PATH}/Train/Train/Rural/masks_png"
]

val_image_dirs = [
    f"{BASE_PATH}/Val/Val/Urban/images_png",
    f"{BASE_PATH}/Val/Val/Rural/images_png"
]

val_mask_dirs = [
    f"{BASE_PATH}/Val/Val/Urban/masks_png",
    f"{BASE_PATH}/Val/Val/Rural/masks_png"
]


In [36]:
def create_tf_dataset(image_dirs, mask_dirs):
    dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(image_dirs, mask_dirs),
        output_signature=(
            tf.TensorSpec(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(IMAGE_SIZE, IMAGE_SIZE), dtype=tf.int32),
            tf.TensorSpec(shape=(IMAGE_SIZE, IMAGE_SIZE), dtype=tf.float32),
        )
    )
    dataset = dataset.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset



train_dataset = create_tf_dataset(train_image_dirs, train_mask_dirs) 
val_dataset = create_tf_dataset(val_image_dirs, val_mask_dirs)


In [37]:
def decode_mask(mask):
    rgb_mask = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    for idx, color in enumerate(PALETTE):
        rgb_mask[mask == idx] = color
    return rgb_mask

In [38]:
def sanity_check(dataset, num_samples=3):
    batch = next(iter(dataset))

    
    if len(batch) == 3:
        images, masks, _ = batch
    else:
        images, masks = batch

    for i in range(min(num_samples, images.shape[0])):
        img = images[i].numpy()
        mask = masks[i].numpy()

        plt.figure(figsize=(10,5))

        plt.subplot(1,2,1)
        plt.imshow((img * 255).astype(np.uint8))
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1,2,2)
        plt.imshow(decode_mask(mask))
        plt.title("Mask")
        plt.axis("off")

        unique_ids = np.unique(mask)
        patches = [
            mpatches.Patch(
                color=PALETTE[j] / 255.0,
                label=CLASS_NAMES[j]
            ) for j in unique_ids
        ]

        plt.legend(handles=patches, bbox_to_anchor=(1.05,1), loc="upper left")
        plt.show()
sanity_check(val_dataset, num_samples=3)

UnknownError: {{function_node __wrapped__IteratorGetNext_output_types_3_device_/job:localhost/replica:0/task:0/device:CPU:0}} FileNotFoundError: [WinError 3] The system cannot find the path specified: '/kaggle/input/loveda-dataset/Val/Val/Urban/images_png'
Traceback (most recent call last):

  File "C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_33616\1171930288.py", line 3, in data_generator
    images = sorted(os.listdir(img_dir))
                    ~

In [ ]:
def unet_model(input_shape=(256,256,3), num_classes=7):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = layers.Conv2D(64, 3, activation="relu", padding="same")(inputs)
    c1 = layers.Conv2D(64, 3, activation="relu", padding="same")(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(128, 3, activation="relu", padding="same")(p1)
    c2 = layers.Conv2D(128, 3, activation="relu", padding="same")(c2)
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    b = layers.Conv2D(256, 3, activation="relu", padding="same")(p2)
    b = layers.Conv2D(256, 3, activation="relu", padding="same")(b)

    # Decoder
    u1 = layers.UpSampling2D()(b)
    u1 = layers.Concatenate()([u1, c2])
    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(u1)
    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(c3)

    u2 = layers.UpSampling2D()(c3)
    u2 = layers.Concatenate()([u2, c1])
    c4 = layers.Conv2D(64, 3, activation="relu", padding="same")(u2)
    c4 = layers.Conv2D(64, 3, activation="relu", padding="same")(c4)

    outputs = layers.Conv2D(num_classes, 1, activation="softmax")(c4)

    return models.Model(inputs, outputs)
model = unet_model()


In [ ]:
class SafeMeanIoU(tf.keras.metrics.MeanIoU):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.argmax(y_pred, axis=-1)
        return super().update_state(y_true, y_pred, sample_weight)


In [ ]:
train_dataset = train_dataset.map(
    lambda x, y, *_: (x, y),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    lambda x, y, *_: (x, y),
    num_parallel_calls=tf.data.AUTOTUNE
)



In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_pred = tf.argmax(y_pred, axis=-1)
    y_true = tf.cast(y_true, tf.int32)
    y_pred = tf.cast(y_pred, tf.int32)

    y_true = tf.one_hot(y_true, NUM_CLASSES)
    y_pred = tf.one_hot(y_pred, NUM_CLASSES)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])

    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth))


model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.MeanIoU(num_classes=NUM_CLASSES),
        dice_coef
    ]
)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        dice_coef,
        SafeMeanIoU(num_classes=NUM_CLASSES)
    ]
)


In [ ]:


history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)


In [ ]:
model.save("model.h5")